# 2. Memory and Context Management

## Overview

Large language models are **stateless**. Every call starts from nothing. The "memory" you
experience in a chatbot is an illusion created by resending previous turns on every request.

That creates the central engineering problem of this notebook: the conversation grows, the
context window does not. This notebook covers how to give an assistant memory and how to keep
that memory from overflowing.

Our example is a **member services conversation**. A member calls about a denied claim, and the
assistant has to hold the thread of who they are and what has already been said — without
letting one member's details leak into another member's session.

You will learn to:

- Prove that the model has no memory of its own
- Add durable memory with a LangGraph **checkpointer**
- Isolate conversations using **threads**
- **Trim** message history to fit a token budget
- **Summarise** old turns instead of discarding them


## Concepts

| Concept | What it does |
|---|---|
| **Short-term memory** | The message list for the current conversation |
| **Long-term memory** | Facts persisted across conversations (a database, a vector store) |
| **Checkpointer** | LangGraph component that saves state after every step |
| **Thread** | A named conversation. State is scoped to a `thread_id` |
| **Trimming** | Dropping old messages to stay within the token budget |
| **Summarisation** | Replacing old messages with a compact summary |
| **Context engineering** | Deciding what to put in the window on each call |

> **Sample data only.** Every member name, ID, claim number and plan detail in this notebook is
> invented for training purposes. Do not paste real member data, real claim numbers, or anything
> else that would count as protected health information into a lab environment.
>
> That is not only a compliance point, it is an architecture point. Notice at the end of this
> notebook that the durable checkpointer writes conversation state to a plain SQLite file on
> disk. In production that file is a store of member conversations and needs the same encryption,
> access control, retention limit and audit trail as any other clinical or claims system. Memory
> is a data-protection decision, not just a UX feature.

#### 0. Setup — connect to Bedrock

In [2]:
# ---------------------------------------------------------------------------
# SageMaker Studio setup
#
# There is no .env file and no access key in this lab. boto3 automatically
# picks up short-lived credentials from the SageMaker execution role attached
# to this space, which is both simpler and safer than a long-lived key pair.
# ---------------------------------------------------------------------------
import os
import boto3

# Leave this as None to use the region this SageMaker space is running in.
# Set it ONLY if you enabled Bedrock model access in a different region.
BEDROCK_REGION = None

AWS_REGION = BEDROCK_REGION or boto3.session.Session().region_name or "us-east-1"
CHAT_MODEL_ID = os.environ.get("BEDROCK_CHAT_MODEL_ID", "amazon.nova-pro-v1:0")

from langchain_aws import ChatBedrockConverse

session = boto3.Session(region_name=AWS_REGION)

# Show which IAM identity this notebook is using. On SageMaker this is the
# execution role of your space - the role that needs Bedrock permissions.
identity = session.client("sts").get_caller_identity()
EXECUTION_ROLE_ARN = identity["Arn"]

model = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    client=session.client("bedrock-runtime"),
    temperature=0,
    max_tokens=1024,
)

MEMBER_SERVICES_PROMPT = (
    "You are a health plan member services assistant. You help with benefits, coverage, "
    "enrolment, premiums, claims and appeals. You never give clinical advice. Be concise."
)

print("Region         :", AWS_REGION)
print("Execution role :", EXECUTION_ROLE_ARN)
print("Connected to   :", CHAT_MODEL_ID)

---
## 1. Proving the model is stateless

Run these two cells in order. The second question refers to the first — and fails.

In [3]:
first = model.invoke(
    "My name is Maria Alvarez, member ID HP-4471, and I am calling about a denied claim."
)
print(first.content)

In [4]:
second = model.invoke("What is my member ID and what am I calling about?")
print(second.content)

The model has no idea. Nothing was carried over. The **only** way it can know is if we
send the earlier turns again.

#### 1.1. Manual memory — resend the history yourself

In [5]:
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage("My name is Maria Alvarez, member ID HP-4471, and I am calling about a denied claim."),
    AIMessage(first.content),
    HumanMessage("What is my member ID and what am I calling about?"),
]

print(model.invoke(history).content)

It works — but now *you* own the list. You have to append every turn, persist it
somewhere, and split it per member. That is exactly what a checkpointer does for you.

---
## 2. Automatic memory with a LangGraph checkpointer

A **checkpointer** saves the agent's state after every step and reloads it on the next call.
You address a saved state with a `thread_id`, which acts as the conversation key.

#### 2.1. In-memory checkpointer

In [6]:
from langchain.agents import create_agent

try:
    from langgraph.checkpoint.memory import InMemorySaver as MemoryCheckpointer
except ImportError:  # older langgraph releases
    from langgraph.checkpoint.memory import MemorySaver as MemoryCheckpointer

checkpointer = MemoryCheckpointer()

agent = create_agent(
    model,
    tools=[],
    system_prompt=MEMBER_SERVICES_PROMPT,
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "member-HP4471-case-88213"}}

r1 = agent.invoke(
    {"messages": [{"role": "user", "content":
                   "My name is Maria Alvarez, member ID HP-4471. My claim for an MRI was denied."}]},
    config,
)
print(r1["messages"][-1].content)

In [7]:
# Same thread_id -> the agent remembers, and we sent no history ourselves
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What was my member ID and which claim did I mention?"}]},
    config,
)
print(r2["messages"][-1].content)

#### 2.2. Threads isolate conversations

Change the `thread_id` and you get a completely separate memory. This is how one deployed
assistant serves thousands of members without leaking context between them.

In this domain that isolation is a **confidentiality control**, not a convenience. A thread key
that collides — two members sharing a session ID, or a reused `thread_id` after a call transfer —
discloses one member's claim history to another. Key threads on something that cannot collide,
and never key them on something a user can set.

In [8]:
other_config = {"configurable": {"thread_id": "member-HP9032-case-88999"}}

r3 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my member ID?"}]},
    other_config,
)
print("Second member's thread ->", r3["messages"][-1].content)

#### 2.3. Inspect what is stored in a thread

In [9]:
state = agent.get_state(config)

print("Messages stored in Maria's thread:", len(state.values["messages"]))
for m in state.values["messages"]:
    print(f"  [{m.type:9}] {m.content[:70]}")

#### 2.4. Durable memory with SQLite

`InMemorySaver` disappears when the kernel restarts. For anything real, persist to disk.
The same agent code works — only the checkpointer changes.

In [10]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("./memory.db", check_same_thread=False)
sqlite_checkpointer = SqliteSaver(conn)

durable_agent = create_agent(
    model,
    tools=[],
    system_prompt=MEMBER_SERVICES_PROMPT,
    checkpointer=sqlite_checkpointer,
)

durable_config = {"configurable": {"thread_id": "case-88213-durable"}}

durable_agent.invoke(
    {"messages": [{"role": "user", "content":
                   "Note for the file: this member is enrolled in the Gold PPO plan and their "
                   "appeal deadline is 60 days from the denial notice."}]},
    durable_config,
)

answer = durable_agent.invoke(
    {"messages": [{"role": "user", "content": "Which plan is this member on and what is the deadline?"}]},
    durable_config,
)
print(answer["messages"][-1].content)
print("\nState is now on disk in ./memory.db - restart the kernel and it survives.")

In [11]:
# Look at what actually landed on disk. This is the cell to remember when someone
# asks where conversation memory lives and who can read it.
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()
print("Tables in memory.db:", [t[0] for t in tables])

rows = conn.execute("SELECT COUNT(*) FROM checkpoints").fetchone()
print("Checkpoint rows stored:", rows[0])
print()
print("In production this file is member conversation data. It needs encryption at rest,")
print("scoped access, a retention policy and an audit trail - the same as any claims store.")

---
## 3. Context window management

Memory that only grows will eventually break: you will hit the context limit, latency rises,
and cost rises with every token you resend. There are two standard fixes.

#### 3.1. Trimming — drop what does not fit

`trim_messages` keeps the most recent messages within a token budget. Note
`include_system=True`, which protects the system prompt from being trimmed away, and
`start_on="human"`, which prevents a dangling tool result at the start of the window.

Protecting the system prompt matters more here than in most domains: that prompt is where your
"no clinical advice" guardrail lives. Trim it away and the assistant quietly loses its scope
limit forty turns into a long call.

In [ ]:
!pip install transformers

In [15]:
from langchain_core.messages import SystemMessage, trim_messages

def count_tokens(messages):
    return sum(len(m.content) for m in messages)

long_history = [SystemMessage("You are an AWS assistant.")]
for i in range(1, 11):
    long_history.append(HumanMessage(f"Question {i}: tell me about AWS service number {i}."))
    long_history.append(AIMessage(f"Answer {i}: here is a description of service {i}."))

print("Before trimming:", len(long_history), "messages")

trimmed = trim_messages(
    long_history,
    max_tokens=300,
    strategy="last",
    token_counter=count_tokens,
    include_system=True,
    start_on="human",
    allow_partial=False,
)

print("After trimming :", len(trimmed), "messages\n")
for m in trimmed:
    print(f"  [{m.type:9}] {m.content[:60]}")

#### 3.2. Trimming inside a chain

Because `trim_messages` is a Runnable, you can pipe it in front of the model so every call is
automatically bounded.

In [17]:
trimmer = trim_messages(
    max_tokens=300,
    strategy="last",
    token_counter=count_tokens,
    include_system=True,
    start_on="human",
)

bounded_chain = trimmer | model

print(bounded_chain.invoke(long_history + [HumanMessage("What was my last question?")]).content)

#### 3.3. Summarisation — compress instead of discard

Trimming throws information away. Summarising keeps the meaning at a fraction of the tokens.
The usual production pattern is a hybrid: summarise everything older than N turns, keep the
last N verbatim.

Read the summariser prompt below carefully. In a member services context the summary becomes
the **case note** — the thing a second agent reads after a transfer. It has to preserve
identifiers, dates, dollar amounts and commitments made to the member, because a summary that
drops "we agreed to expedite the appeal" is worse than no summary at all.

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You compress a member services conversation into a case note. Preserve every "
     "identifier, date, dollar amount, decision made, commitment given to the member, and "
     "open question. Do not add anything that was not said. Maximum 100 words."),
    ("human", "Conversation to compress:\n\n{conversation}"),
])

summariser = summary_prompt | model | StrOutputParser()


def compress(messages, keep_last=4):
    """Summarise everything except the most recent `keep_last` messages."""
    system = [m for m in messages if m.type == "system"]
    body = [m for m in messages if m.type != "system"]

    if len(body) <= keep_last:
        return messages

    older, recent = body[:-keep_last], body[-keep_last:]
    transcript = "\n".join(f"{m.type}: {m.content}" for m in older)
    summary = summariser.invoke({"conversation": transcript})

    return system + [SystemMessage(f"Case note from earlier in this call: {summary}")] + recent


compressed = compress(long_history, keep_last=4)

print("Original  :", len(long_history), "messages")
print("Compressed:", len(compressed), "messages\n")
for m in compressed:
    print(f"  [{m.type:9}] {m.content[:100]}")

> **A failure mode worth testing for.** Summarisation is lossy by design, and what it drops is
> chosen by a model, not by you. Before shipping this, build a small evaluation set of real case
> types and check that identifiers, deadlines and commitments survive compression. "The summary
> reads well" is not the acceptance criterion; "the appeal deadline is still in it" is.

---
## 4. Context engineering

"Memory" is one input among several. On every model call you are assembling a context window
out of competing demands:

```
+---------------------------------------------------+
|  System prompt        (who the assistant is,      |  always
|                        and what it must refuse)   |
+---------------------------------------------------+
|  Long-term memory     (this member's plan, tier)  |  retrieved
+---------------------------------------------------+
|  Conversation summary (earlier in the call)       |  generated
+---------------------------------------------------+
|  Recent messages      (last N turns, verbatim)    |  kept
+---------------------------------------------------+
|  Retrieved documents  (plan documents, RAG)       |  retrieved
+---------------------------------------------------+
|  Tool results         (eligibility, claim status) |  live
+---------------------------------------------------+
```

Practical guidance:

- **Put instructions first, data last.** Models weight the beginning and end most heavily.
- **Retrieve, do not dump.** Four relevant chunks beat forty mediocre ones.
- **Give every piece of context a source label**, so the model can cite it and you can debug it.
- **Measure.** Log token counts per call; context bloat shows up in the bill before it shows up
  in the output quality.
- **Put the minimum member data in the window.** Every identifier you add is one more thing that
  ends up in a log, a trace and a checkpoint file. If the assistant does not need a date of
  birth to answer, do not include it.

#### 4.1. Assembling a context window by hand

In [19]:
def build_context(system_instruction, member_facts, conversation_summary, recent, question):
    parts = [SystemMessage(system_instruction)]
    if member_facts:
        parts.append(SystemMessage("Known facts about this member:\n" + "\n".join(member_facts)))
    if conversation_summary:
        parts.append(SystemMessage("Case note from earlier: " + conversation_summary))
    parts.extend(recent)
    parts.append(HumanMessage(question))
    return parts


context = build_context(
    system_instruction=MEMBER_SERVICES_PROMPT,
    member_facts=[
        "Plan: Gold PPO",
        "Annual deductible: $1,500, of which $1,500 is met",
        "Prior authorisation required for advanced imaging",
    ],
    conversation_summary=(
        "Member's MRI claim was denied for missing prior authorisation. "
        "Denial notice dated 12 March. Appeal window is 60 days."
    ),
    recent=[HumanMessage("The provider says they did submit the authorisation request.")],
    question="Given all of the above, what should my next step be?",
)

for m in context:
    print(f"[{m.type:9}] {m.content[:80]}")

print("\n--- MODEL ANSWER ---")
print(model.invoke(context).content)

Notice how much of the quality of that answer came from the **facts block**, not from the
model. The deductible being met, the prior authorisation rule and the appeal window are what
turn a generic answer into an actionable one. That is context engineering: most of the work is
deciding what to put in the window, not how to phrase the question.

---
## Summary

- LLMs are stateless; memory is history you resend
- A **checkpointer** automates that, keyed by `thread_id`
- Thread isolation is a confidentiality control — never key it on something a user controls
- `InMemorySaver` for experiments, `SqliteSaver` for durable state
- Durable memory is member data on disk: encrypt it, scope it, expire it, audit it
- **Trim** to enforce a token budget, **summarise** to keep meaning at lower cost
- Protect the system prompt from trimming, or you lose your scope guardrail mid-call
- Context engineering is about what you *include*, not just how much fits

**Next:** notebook `3-RAG-Pipeline.ipynb`, where retrieved plan documents become the most
important thing in the context window.